[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C67_LLM_Judge_Course/01_judge_design/01_judge_design.ipynb)

# 01 · Judge 设计（分布压缩 / 平局口径 / rubric 分解 / 参考答案 / 结构化输出 / 锚点）

目标：把「怎么写一个 judge」从调 prompt 的手艺，变成几个**可以测量、可以对比**的设计决策。

本 notebook 你会亲手实现：
1. **分数分布压缩的度量** —— 熵与有效档位数，量化「五档量表实际只用了一档半」
2. **三种缓解手段的对比** —— rubric 分解 / 锚点 / 更细量表，谁真的把判别力救回来了
3. **平局的三种统计口径** —— 同一批判断，三个不同的胜率
4. **rubric 分解 vs 整体分** —— 分辨力对比 + 权重敏感性 + 否决项
5. **参考答案的双刃效果** —— 一致性提升，但「像参考」会被误当成「正确」
6. **结构化输出的解析失败率** —— 为什么悄悄丢掉解析失败的样本会让一致率虚高

> 心智模型：**设计 judge 的第一步不是写 prompt，是把问题往可判定的方向改写。
> 能拆成 rubric 就不要打整体分，能比较就不要打绝对分。**

## 1 · 分数分布压缩：量化「五档量表实际用了几档」

In [ ]:
import math, json, re, hashlib
from collections import Counter, defaultdict
import numpy as np

def score_entropy(scores, scale=5):
    """分数分布的熵（bit）。"""
    cnt = Counter(np.asarray(scores).astype(int).tolist())
    n = sum(cnt.values())
    return -sum((c / n) * math.log2(c / n) for c in cnt.values() if c > 0)

def effective_levels(scores, scale=5):
    """有效档位数 = 2^H。五档量表的上限是 5；实际常常只有 1.5-2.5。"""
    return 2 ** score_entropy(scores, scale)

rng = np.random.default_rng(0)
N = 2000
q = rng.uniform(0, 1, N)                       # 真实质量

def judge_pointwise(q, noise=0.6, compression=0.55, severity=0.0, scale=5, rng=None):
    rng = rng or np.random.default_rng(0)
    v = q + rng.normal(0, noise, size=np.shape(q)) - severity
    v = 0.5 + compression * (v - 0.5)
    return np.clip(np.round(v * (scale - 1) + 1), 1, scale)

s_compressed = judge_pointwise(q, noise=0.25, compression=0.30, rng=np.random.default_rng(1))
print('分数分布:', dict(sorted(Counter(s_compressed.astype(int).tolist()).items())))
print(f'熵 {score_entropy(s_compressed):.3f} bit | 有效档位数 {effective_levels(s_compressed):.2f} / 5')
assert effective_levels(s_compressed) < 2.5
print('\n✅ 五档量表实际只用到了不到 2.5 档——')
print('   落在同一档里的样本之间，这个 judge 提供的信息是 0 bit。')

In [ ]:
# 判别力：分数与真实质量的相关系数（判别力的直接度量）
def discrimination(scores, q_true):
    s = np.asarray(scores, dtype=float)
    t = np.asarray(q_true, dtype=float)
    sc, tc = s - s.mean(), t - t.mean()
    d = math.sqrt(float((sc ** 2).sum()) * float((tc ** 2).sum()))
    return float((sc * tc).sum() / d) if d else 0.0

VARIANTS = {
    '基线（压缩严重）': dict(noise=0.25, compression=0.30, scale=5),
    '加锚点（拉开分布）': dict(noise=0.25, compression=0.85, scale=5),
    '0-100 细量表':      dict(noise=0.25, compression=0.55, scale=101),
    '降噪（更好的 prompt）': dict(noise=0.12, compression=0.55, scale=5),
}
print(f"{'变体':<22}{'有效档位':>10}{'与真值相关':>12}")
for name, kw in VARIANTS.items():
    sc = judge_pointwise(q, rng=np.random.default_rng(5), **kw)
    lv = effective_levels(sc, kw['scale'])
    print(f'{name:<22}{lv:>10.2f}{discrimination(sc, q):>12.3f}')

base = judge_pointwise(q, noise=0.25, compression=0.30, scale=5, rng=np.random.default_rng(5))
anchored = judge_pointwise(q, noise=0.25, compression=0.85, scale=5, rng=np.random.default_rng(5))
assert discrimination(anchored, q) > discrimination(base, q)
print('\n✅ 注意 0-100 细量表这一行：有效档位数暴涨，但与真值的相关只涨了一点点——')
print('   **把量表变细并不会凭空创造信息**，它只是把同样的噪声铺得更开。')
print('   真正有效的是「拉开分布」（锚点）和「降噪」（更好的 prompt / rubric）。')

## 2 · rubric 分解：为什么它比整体分更有判别力

把一个整体判断拆成 K 个独立的二元判断，再平均。每个子判断有各自的噪声，
但**独立噪声在平均后按 $1/\sqrt{K}$ 衰减**——这就是 rubric 的全部数学。

In [ ]:
def holistic_judge(q, noise=0.25, compression=0.30, rng=None):
    return judge_pointwise(q, noise=noise, compression=compression, scale=5, rng=rng)

def rubric_judge(q, k_items=5, item_noise=0.35, weights=None, rng=None):
    """K 个独立的二元 rubric 项：第 j 项通过的真实概率与 q 相关，judge 各自带噪声地判定。"""
    rng = rng or np.random.default_rng(0)
    q = np.asarray(q, dtype=float)
    w = np.ones(k_items) if weights is None else np.asarray(weights, dtype=float)
    w = w / w.sum()
    total = np.zeros_like(q)
    for j in range(k_items):
        thresh = (j + 0.5) / k_items                     # 每项的难度不同
        perceived = q + rng.normal(0, item_noise, size=q.shape)
        total += w[j] * (perceived > thresh).astype(float)
    return total

h = holistic_judge(q, rng=np.random.default_rng(9))
r = rubric_judge(q, k_items=5, rng=np.random.default_rng(9))
print(f'整体分     与真值相关 {discrimination(h, q):.3f} | 有效档位 {effective_levels(h):.2f}')
print(f'rubric(5项) 与真值相关 {discrimination(r, q):.3f} | 取值个数 {len(set(np.round(r,6).tolist()))}')
assert discrimination(r, q) > discrimination(h, q)

print(f"\n{'rubric 项数':>12}{'与真值相关':>12}")
for k in [1, 3, 5, 8, 12]:
    rr = rubric_judge(q, k_items=k, rng=np.random.default_rng(9))
    print(f'{k:>12}{discrimination(rr, q):>12.3f}')
print('\n✅ 项数越多相关越高，但收益递减（$1/\\sqrt{K}$）。')
print('   实践中 4-6 项就能拿到大部分收益——再往上加，边际收益抵不过设计与调用成本。')

In [ ]:
# 权重敏感性：结论会不会被权重扰动翻转
def weight_sensitivity(qA, qB, k_items=5, delta=0.2, n=300, seed=0):
    rng = np.random.default_rng(seed)
    wins = 0
    for _ in range(n):
        w = np.ones(k_items) * (1 + rng.uniform(-delta, delta, k_items))
        a = rubric_judge(qA, k_items, weights=w, rng=np.random.default_rng(1)).mean()
        b = rubric_judge(qB, k_items, weights=w, rng=np.random.default_rng(2)).mean()
        if a > b:
            wins += 1
    return wins / n

rng2 = np.random.default_rng(3)
qA = rng2.uniform(0, 1, 400) * 0.0 + rng2.uniform(0.30, 0.75, 400)
qB = rng2.uniform(0.28, 0.73, 400)
frac = weight_sensitivity(qA, qB)
print(f'权重扰动 ±20% 后，A 胜出的比例: {frac:.1%}')
assert 0.0 <= frac <= 1.0
verdict = '稳健' if frac > 0.9 or frac < 0.1 else '结论由权重决定，不该写成「A 更好」'
print(f'判读: {verdict}')
print('\n✅ 敏感性分析是 rubric 聚合的必备步骤——')
print('   胜出比例接近 50% 时，真正的分歧点在权重上而不在数据上。')

In [ ]:
# 否决项：某些问题应当直接归零，而不是按权重扣分
def rubric_with_veto(rubric_score, veto_flags):
    """veto_flags 为 True 的样本（编造引用 / 承诺做不到的事 / 违反安全策略）直接归零。"""
    return np.where(np.asarray(veto_flags, dtype=bool), 0.0, np.asarray(rubric_score, dtype=float))

rng3 = np.random.default_rng(4)
veto = rng3.random(len(q)) < 0.06          # 6% 的样本触发否决项
r_veto = rubric_with_veto(r, veto)
print(f'无否决项 平均分 {r.mean():.3f} | 有否决项 平均分 {r_veto.mean():.3f}')
print(f'被否决样本在无否决口径下的平均分: {r[veto].mean():.3f}（看起来还不错）')
assert r_veto.mean() < r.mean()
assert r[veto].mean() > 0.3
print('\n✅ 关键在最后一行：被否决的样本在加权口径下平均分并不低——')
print('   一个「编造了引用但其他方面都写得很好」的回答，按权重扣分后仍然是高分。')
print('   **这类问题必须用否决项处理，不能靠权重。**')

## 3 · 平局的三种统计口径：同一批判断，三个胜率

In [ ]:
def win_rate(w, l, t, mode='half'):
    total = w + l + t
    if mode == 'half':      # 平局算半分（推荐，与 Bradley-Terry 一致）
        return (w + 0.5 * t) / total if total else float('nan')
    if mode == 'drop':      # 丢弃平局
        return w / (w + l) if (w + l) else float('nan')
    if mode == 'loss':      # 平局算输（偶尔在"必须严格更好"的场景用）
        return w / total if total else float('nan')
    raise ValueError(mode)

CASES = [('平局少', 120, 100, 20), ('平局中等', 100, 80, 120), ('平局很多', 60, 40, 200)]
print(f"{'场景':<12}{'W':>5}{'L':>5}{'T':>5}{'half':>10}{'drop':>10}{'loss':>10}")
for name, w, l, t in CASES:
    print(f'{name:<12}{w:>5}{l:>5}{t:>5}'
          f'{win_rate(w,l,t,"half"):>10.1%}{win_rate(w,l,t,"drop"):>10.1%}{win_rate(w,l,t,"loss"):>10.1%}')

h1 = win_rate(60, 40, 200, 'half')
d1 = win_rate(60, 40, 200, 'drop')
assert abs(d1 - h1) > 0.05
print(f'\n平局率 {200/300:.0%} 时，两种口径差 {abs(d1-h1):.1%}——')
print('✅ 「胜率 57%」这句话在没写明平局口径时是不完整的。')
print('   推荐用 half（与 BT 模型一致，且不会因平局多而人为放大差距）。')

## 4 · 参考答案的双刃效果

参考答案提升一致性，但会让 judge 奖励「像参考」而不是「同样正确」。
构造一批「正确但与参考风格不同」的回答，看它们被怎么对待。

In [ ]:
rng = np.random.default_rng(12)
M = 1500
correct = (rng.random(M) < 0.5).astype(float)          # 是否真的正确
similarity = rng.uniform(0, 1, M)                       # 与参考答案的表面相似度（与正确性独立）

def judge_with_reference(correct, similarity, w_correct=1.0, w_sim=0.0, noise=0.35, seed=0):
    rng = np.random.default_rng(seed)
    v = w_correct * correct + w_sim * similarity + rng.normal(0, noise, size=correct.shape)
    return (v > (w_correct + w_sim) / 2).astype(int)

no_ref = judge_with_reference(correct, similarity, w_correct=1.0, w_sim=0.0, noise=0.55, seed=1)
with_ref = judge_with_reference(correct, similarity, w_correct=1.0, w_sim=0.0, noise=0.28, seed=2)
with_ref_leak = judge_with_reference(correct, similarity, w_correct=1.0, w_sim=0.6, noise=0.28, seed=3)

for name, pred in [('无参考', no_ref), ('有参考（理想）', with_ref), ('有参考（相似度泄漏）', with_ref_leak)]:
    acc = float((pred == correct).mean())
    # 「正确但不像参考」的子群上，被判对的比例
    sub = (correct == 1) & (similarity < 0.3)
    recall_unlike = float(pred[sub].mean())
    print(f'{name:<22} 总一致率 {acc:.1%} | 「正确但不像参考」被判对的比例 {recall_unlike:.1%}')

acc_leak = float((with_ref_leak == correct).mean())
acc_ref = float((with_ref == correct).mean())
sub = (correct == 1) & (similarity < 0.3)
assert with_ref_leak[sub].mean() < with_ref[sub].mean()
print('\n✅ 「相似度泄漏」这一行是关键：总一致率看起来还行，')
print('   但「用不同方法做对了」的回答被系统性地判错——')
print('   这与 C66-03「把轨迹相似度当主指标等于惩罚探索」是完全同构的错误。')
print('   缓解：prompt 里显式写「参考只是一种正确解法」+ rubric 用「结论是否正确」而非「是否与参考一致」。')

## 5 · 结构化输出：解析失败率与选择偏倚

关键洞察：**解析失败不是随机的**——难判的样本更容易输出一堆解释而不是干净 JSON。
悄悄丢掉它们，剩下样本的一致率会虚高。

In [ ]:
def parse_verdict(raw):
    """宽松解析：剥离代码块、找第一个 JSON 对象、归一化 enum。"""
    if raw is None:
        return None
    txt = re.sub(r'^```(?:json)?|```$', '', raw.strip(), flags=re.M).strip()
    m = re.search(r'\{.*\}', txt, flags=re.S)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None
    v = str(obj.get('verdict', '')).strip().lower()
    for key, out in [('tie', 'tie'), ('a', 'A'), ('b', 'B')]:
        if v.startswith(key):
            return out
    return None

SAMPLES = [
    '{"reasoning": "A is more accurate.", "verdict": "A"}',
    '```json\n{"reasoning": "…", "verdict": "B"}\n```',
    'Sure! Here is my analysis:\n{"reasoning": "…", "verdict": "tie"}',
    '{"reasoning": "A is better because it',                 # 截断
    'I think A is better, but it depends.',                  # 根本没给 JSON
    '{"reasoning": "…", "verdict": "A is better"}',          # 枚举越界（宽松解析可救）
]
for s_ in SAMPLES:
    print(f'{parse_verdict(s_)!s:<6} <- {s_[:52]!r}')
assert parse_verdict(SAMPLES[0]) == 'A'
assert parse_verdict(SAMPLES[1]) == 'B'
assert parse_verdict(SAMPLES[2]) == 'tie'
assert parse_verdict(SAMPLES[3]) is None and parse_verdict(SAMPLES[4]) is None
assert parse_verdict(SAMPLES[5]) == 'A'
print('\n✅ 宽松解析能救回代码块包裹与枚举越界，但救不了截断与完全没给 JSON。')

In [ ]:
# 解析失败与判断难度相关 → 悄悄丢弃会让一致率虚高
rng = np.random.default_rng(17)
K = 3000
difficulty = rng.uniform(0, 1, K)                     # 越大越难判
truth_v = rng.integers(0, 2, K)
# 难样本更容易判错
pred = np.where(rng.random(K) < 0.10 + 0.35 * difficulty, 1 - truth_v, truth_v)
# 也更容易解析失败
parse_ok = rng.random(K) > (0.01 + 0.20 * difficulty)

acc_all = float((pred == truth_v).mean())
acc_parsed = float((pred[parse_ok] == truth_v[parse_ok]).mean())
acc_with_tie = float(np.where(parse_ok, pred == truth_v, 0.5).mean())   # 失败记平局计入分母
print(f'解析成功率            {parse_ok.mean():.1%}')
print(f'全体一致率（真值）     {acc_all:.1%}')
print(f'只统计解析成功的样本   {acc_parsed:.1%}   ← 虚高')
print(f'失败记平局并计入分母   {acc_with_tie:.1%}')
assert acc_parsed > acc_all
print('\n✅ 悄悄丢掉解析失败的样本，一致率虚高了几个点。')
print('   正确做法：重试；重试仍失败的记为平局并**计入分母**，同时把解析失败率写进报告。')

## ✏️ 练习 1：有效档位数与「该不该换量表」

实现 `scale_utilization(scores, scale)`：返回 `有效档位数 / 名义档位数`。
用它判断：一个 0–100 的量表如果分数全部落在 70/75/80/85 四个值上，利用率是多少。

In [ ]:
def scale_utilization(scores, scale):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
uniform5 = np.array([1, 2, 3, 4, 5] * 40)
assert abs(scale_utilization(uniform5, 5) - 1.0) < 1e-9
const = np.array([4] * 200)
assert abs(scale_utilization(const, 5)) < 1e-9 or scale_utilization(const, 5) == 0.2
lumpy100 = np.array([70, 75, 80, 85] * 50)
u = scale_utilization(lumpy100, 101)
print(f'0-100 量表但只用了 4 个值 → 利用率 {u:.2%}')
assert u < 0.05
print(f'五档均匀分布 → 利用率 {scale_utilization(uniform5, 5):.0%}')
print('✅ 练习 1 通过：把量表变细而分数仍然扎堆，利用率会低到离谱——')
print('   这个数字是「该不该换设计」的直接信号。')

## ✏️ 练习 2：rubric 项数的边际收益

实现 `marginal_gain(k, item_noise=0.35, n=2000, seed=0)`：
返回从 `k` 项加到 `k+1` 项时，与真值相关系数的增量。
用它找出「加到第几项之后收益就不值得了」。

In [ ]:
def marginal_gain(k, item_noise=0.35, n=2000, seed=0):
    # TODO：用上面已定义的 rubric_judge 与 discrimination
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
g1 = marginal_gain(1)
g8 = marginal_gain(8)
assert g1 > g8, '边际收益必须递减'
print(f"{'k→k+1':>8}{'相关系数增量':>14}")
for k in [1, 2, 3, 5, 8, 12]:
    print(f'{f"{k}→{k+1}":>8}{marginal_gain(k):>14.4f}')
print('✅ 练习 2 通过：从 1 项加到 2 项收益巨大，从 8 项加到 9 项几乎没有——')
print('   4-6 项是性价比的甜点区。')

## ✏️ 练习 3：平局口径的转换

实现 `convert_win_rate(wr_drop, tie_rate)`：已知「丢弃平局口径」下的胜率 `wr_drop`
与平局比例 `tie_rate`，反算「平局算半分」口径下的胜率。

推导：设总数为 1，则 $W+L = 1-t$，$W = (1-t)\cdot wr_{drop}$，
所以 $wr_{half} = W + t/2 = (1-t)\cdot wr_{drop} + t/2$。

In [ ]:
def convert_win_rate(wr_drop, tie_rate):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert abs(convert_win_rate(0.60, 0.0) - 0.60) < 1e-12
assert abs(convert_win_rate(0.60, 1.0) - 0.50) < 1e-12       # 全是平局 → 必然 50%
w, l, t = 60, 40, 200
assert abs(convert_win_rate(win_rate(w, l, t, 'drop'), t / (w + l + t))
           - win_rate(w, l, t, 'half')) < 1e-9
for tr in [0.0, 0.2, 0.5, 0.8]:
    print(f'丢弃口径 65% + 平局率 {tr:.0%} → half 口径 {convert_win_rate(0.65, tr):.1%}')
print('✅ 练习 3 通过：平局率越高，两个口径差得越远——')
print('   平局会把胜率往 50% 拉，这正是它应该做的（分不出就别装作分得出）。')

## ✏️ 练习 4：解析失败的正确记账

实现 `agreement_with_failures(pred, truth, parse_ok, policy)`，
`policy ∈ {'drop', 'tie', 'retry'}`：
- `drop`：只统计解析成功的样本（会虚高）
- `tie`：失败记平局，按 0.5 分计入分母
- `retry`：失败样本按 `retry_success=0.7` 的概率重试成功（成功后沿用 `pred`），
  仍失败的按 `tie` 处理

返回一致率。`retry` 模式用固定 seed 保证可复现。

In [ ]:
def agreement_with_failures(pred, truth, parse_ok, policy, retry_success=0.7, seed=0):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
a_drop = agreement_with_failures(pred, truth_v, parse_ok, 'drop')
a_tie = agreement_with_failures(pred, truth_v, parse_ok, 'tie')
a_retry = agreement_with_failures(pred, truth_v, parse_ok, 'retry', seed=1)
print(f'drop  {a_drop:.1%}  ← 虚高（悄悄丢掉了难样本）')
print(f'tie   {a_tie:.1%}  ← 最保守')
print(f'retry {a_retry:.1%}  ← 推荐：先重试，仍失败才记平局')
assert a_drop > a_tie
assert a_tie <= a_retry <= a_drop + 1e-9
print('✅ 练习 4 通过：三种记账口径的排序永远是 tie ≤ retry ≤ drop。')
print('   报告里必须写明用的是哪种，并且把解析失败率一起报出来。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def scale_utilization(scores, scale):
    if scale <= 1:
        return 0.0
    return effective_levels(scores, scale) / scale

In [ ]:
# 练习 2 参考答案
def marginal_gain(k, item_noise=0.35, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    qq = rng.uniform(0, 1, n)
    a = discrimination(rubric_judge(qq, k_items=k, item_noise=item_noise,
                                    rng=np.random.default_rng(seed + 1)), qq)
    b = discrimination(rubric_judge(qq, k_items=k + 1, item_noise=item_noise,
                                    rng=np.random.default_rng(seed + 1)), qq)
    return b - a

In [ ]:
# 练习 3 参考答案
def convert_win_rate(wr_drop, tie_rate):
    return (1 - tie_rate) * wr_drop + tie_rate / 2

In [ ]:
# 练习 4 参考答案
def agreement_with_failures(pred, truth, parse_ok, policy, retry_success=0.7, seed=0):
    pred = np.asarray(pred)
    truth = np.asarray(truth)
    ok = np.asarray(parse_ok, dtype=bool)
    hit = (pred == truth).astype(float)
    if policy == 'drop':
        return float(hit[ok].mean())
    if policy == 'tie':
        return float(np.where(ok, hit, 0.5).mean())
    if policy == 'retry':
        rng = np.random.default_rng(seed)
        recovered = (~ok) & (rng.random(len(pred)) < retry_success)
        final_ok = ok | recovered
        return float(np.where(final_ok, hit, 0.5).mean())
    raise ValueError(policy)

---
## 🧪 真实工程胶囊：可直接用的 judge 模板与版本管理

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════
# A. rubric 型 judge 的完整模板（推荐作为默认起点）
# ══════════════════════════════════════════════════════════════════
RUBRIC_JUDGE = "
".join([
    "Evaluate the response against the rubric below.",
    "",
    "<request>{request}</request>",
    "<response>{response}</response>",
    '<reference note="ONE valid solution. Other correct approaches score equally.">{reference}</reference>',
    "",
    "For EACH item, output pass/fail independently. Do not let one item influence another.",
    "",
    "RUBRIC",
    "  R1 answers_question   The response addresses what was actually asked.",
    "  R2 factually_correct  No incorrect factual claims.",
    "  R3 cites_policy       Cites the applicable policy clause when one exists.",
    "  R4 no_overpromise     Does not commit to anything outside stated capabilities.",
    "  R5 actionable         The user can act on it without asking a follow-up.",
    "",
    "VETO (if ANY is true, total score is 0 regardless of the rubric)",
    "  V1 fabricated_citation  Cites a source/clause that does not exist.",
    "  V2 unsafe               Violates the safety policy.",
    "",
    "Return JSON only, reasoning BEFORE verdicts:",
    '{{"reasoning": "<2-3 sentences>",',
    '  "rubric": {{"R1": true, "R2": true, "R3": false, "R4": true, "R5": true}},',
    '  "veto":   {{"V1": false, "V2": false}}}}',
])

def score_from_rubric(obj, weights=None):
    if any(obj["veto"].values()):
        return 0.0                                   # 否决项：直接归零，不按权重扣
    items = obj["rubric"]
    w = weights or {k: 1.0 for k in items}
    tot = sum(w.values())
    return sum(w[k] for k, v in items.items() if v) / tot

# ══════════════════════════════════════════════════════════════════
# B. judge prompt 的版本指纹（判分器也是 harness 的一部分）
# ══════════════════════════════════════════════════════════════════
import hashlib, json
def judge_fingerprint(template, anchors, model_id, temperature):
    payload = json.dumps({"template": template, "anchors": anchors,
                          "model": model_id, "temperature": temperature},
                         sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode()).hexdigest()[:8]
# 把它写进每一条 judge 结果。改了 prompt、改了锚点、换了 judge 模型 → 指纹变 →
# 与历史分数不可直接比较（C68 的 CI 门禁里这是一行 assert）。

# ══════════════════════════════════════════════════════════════════
# C. 结构化输出：优先用原生约束，宽松解析只作兜底
# ══════════════════════════════════════════════════════════════════
# Anthropic / OpenAI 都支持用工具调用或 response_format 强制 schema。
# 无论用哪种，都要记录：parse_ok、retry_count、raw_output（失败样本必须留原文，
# 否则你永远查不出它为什么失败）。

# ══════════════════════════════════════════════════════════════════
# D. 锚点自检（便宜、快、经常抓到问题）
# ══════════════════════════════════════════════════════════════════
# 把每个锚点样例本身当作待评样本送进 judge：
#   5 分锚点被打成 4 分 → prompt 文字描述与锚点样例互相矛盾，模型更信样例
#   → 改锚点比改文字描述有效
'''
print(RECIPE)

### 小结

| 你学到的 | 一句话 | 用在哪 |
|---|---|---|
| 可判定性阶梯 | 设计 judge 的第一步是把问题往可判定方向改写 | 任何新 judge |
| 分布压缩 | 五档量表常常只有效用到一档半；把量表变细不创造信息 | pointwise |
| rubric 分解 | 独立噪声按 1/sqrt(K) 衰减；4–6 项是甜点区 | 提升判别力 |
| 否决项 | 编造引用这类问题必须归零，不能按权重扣分 | rubric 设计 |
| 平局口径 | 平局算半分（与 BT 一致），且必须写明 | pairwise 报告 |
| 参考答案 | 提升一致性，但会奖励「像参考」而非「同样正确」 | 有标准答案的任务 |
| 解析失败 | 失败与难度相关，悄悄丢弃会让一致率虚高 | 工程记账 |
| prompt 版本化 | judge prompt 是 harness 的一部分 | 长期可比性 |

下一模块：**02 · Judge 偏差与去偏**——位置、长度、自偏好、风格四个偏差的量化探针，
以及把它们从读数里减掉的具体方法。